<a href="https://colab.research.google.com/github/fedeiria/tpi-analisis-de-datos/blob/main/notebooks/TPI_Analisis_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trabajo Práctico Integrador — Introducción al Análisis de Datos
## Predicción de cancelación de clientes (Churn) en una empresa de telecomunicaciones

**UTN — Facultad Regional Avellaneda** · 2º Cuatrimestre 2026 · Docente: Enzo Martín Zotti

| Integrante | División |
|---|---|
| Nombre Apellido | — |
| Nombre Apellido | — |
| Nombre Apellido | — |
| Nombre Apellido | — |

## Índice

0. Configuración del entorno
1. **A — Fundamentos** (problema de negocio, hipótesis, clasificación de variables)
2. **A — Manipulación de datos** (selección, filtrado, agregación, transformación)
3. **A — Limpieza y calidad de datos** (nulos, dtypes, outliers)
4. **B — Análisis exploratorio y estadística aplicada**
5. **B — Visualización de datos**
6. **C — Preprocesamiento para modelado**
7. **C — Regresión lineal simple**
8. **C — Clasificación (Churn)**
9. Conclusiones

> **Convención del equipo:** cada sección indica su responsable. Cada gráfico, tabla o resultado va seguido de una celda markdown con la interpretación orientada al negocio.

---
# 0. Configuración del entorno

In [ ]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (r2_score, mean_absolute_error,
                             confusion_matrix, ConfusionMatrixDisplay, accuracy_score)

# Configuración general
RANDOM_STATE = 42          # semilla única para todo el notebook (reproducibilidad)
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)

print("pandas", pd.__version__, "| numpy", np.__version__, "| seaborn", sns.__version__)

### Carga del dataset

Fuente original: [Telco Customer Churn — Kaggle](https://www.kaggle.com/datasets/blastchar/telco-customer-churn).
El CSV se versiona dentro del repositorio (`data/raw/`) y se lee desde la URL *raw* de GitHub, para que el notebook corra en Colab sin credenciales de Kaggle.

In [ ]:
# URL del CSV dentro de nuestro repositorio (completar cuando el repo exista)
DATA_URL = "https://raw.githubusercontent.com/fedeiria/tpi-analisis-de-datos/main/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Copia pública de respaldo del mismo dataset (repositorio de IBM)
FALLBACK_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

try:
    df_raw = pd.read_csv(DATA_URL)
    print("Dataset cargado desde el repositorio del grupo.")
except Exception:
    df_raw = pd.read_csv(FALLBACK_URL)
    print("Dataset cargado desde la URL de respaldo.")

df = df_raw.copy()   # trabajamos sobre una copia; df_raw queda intacto como referencia
print(df.shape)
df.head()

---
# 1. A — Fundamentos
**Responsable:** Federico Iriarte

### 1.1 Problema de negocio

Una empresa de telecomunicaciones ofrece telefonía, internet y servicios adicionales (seguridad, soporte técnico, streaming) bajo distintos tipos de contrato. Una parte de sus clientes **cancela el servicio** y se va a la competencia: a este fenómeno se lo conoce como **churn** (abandono o fuga de clientes).

El churn impacta directamente en los ingresos recurrentes de la empresa y, además, conseguir un cliente nuevo suele ser bastante más caro que retener a uno existente (publicidad, promociones de bienvenida, costos de instalación). Por eso, a la empresa le conviene **detectar a tiempo qué clientes tienen mayor riesgo de irse**, para dirigirles acciones de retención (descuentos, migración a contratos más largos, mejoras en el servicio) antes de que cancelen.

**Pregunta de negocio:** ¿qué características de los clientes y de sus contratos están asociadas a la cancelación, y es posible anticipar qué clientes van a cancelar a partir de esos datos?

### 1.2 Variable objetivo

La variable objetivo es **`Churn`**, que indica si el cliente canceló el servicio en el último mes. Es una variable **cualitativa nominal dicotómica** (dos categorías: `Yes` / `No`). Para el modelado se codificará como binaria (`1` = canceló, `0` = permanece).

In [ ]:
# Distribución de la variable objetivo: frecuencia absoluta y relativa
churn_dist = pd.DataFrame({
    "clientes": df["Churn"].value_counts(),
    "porcentaje": (df["Churn"].value_counts(normalize=True) * 100).round(2),
})
churn_dist

**Interpretación:** aproximadamente **1 de cada 4 clientes (26,5%) canceló** el servicio. Las clases están **desbalanceadas** (73,5% vs. 26,5%): esto se tendrá en cuenta al particionar los datos (partición estratificada) y al interpretar la accuracy del modelo de clasificación, ya que un modelo que predijera siempre "No" acertaría el 73,5% de las veces sin aprender nada útil.

### 1.3 Hipótesis analíticas

A partir del conocimiento del negocio se plantean las siguientes hipótesis, que se contrastarán con los datos en la sección 4:

- **H1 — Tipo de contrato:** los clientes con contrato **mes a mes** cancelan en mayor proporción que los de contratos de uno o dos años, porque no tienen un compromiso de permanencia que los retenga.
- **H2 — Antigüedad:** los clientes con **menor antigüedad** (`tenure`) cancelan más, porque todavía no consolidaron una relación con la empresa.
- **H3 — Método de pago:** los clientes que pagan con **cheque electrónico** cancelan más que los que tienen pagos automáticos (transferencia o tarjeta), porque el pago manual implica una decisión activa cada mes.
- **H4 — Cargo mensual:** los clientes con **cargos mensuales más altos** cancelan más, porque perciben un mayor costo y tienen más incentivo para buscar alternativas.

### 1.4 Clasificación del dataset

El dataset es **estructurado**: los datos están organizados en una tabla con filas y columnas de esquema fijo, donde cada **fila representa un cliente** (registro) y cada **columna un atributo** con un significado y tipo definidos. Se almacena en formato CSV y puede manipularse directamente con Pandas o SQL.

No es *semi-estructurado* (como JSON o XML, donde los registros pueden tener distintos campos o anidamientos) ni *no estructurado* (como texto libre, imágenes o audio, que no tienen un esquema tabular predefinido).

In [ ]:
# Vista general de la estructura: filas, columnas, dtypes y nulos detectados por Pandas
df.info()

**Interpretación:** el dataset tiene **7.043 registros y 21 columnas**, y Pandas no detecta valores nulos explícitos. Sin embargo, la columna `TotalCharges` (un importe en dinero) fue leída como **texto** en lugar de número, lo que anticipa un problema de calidad que se analiza en la sección 3.

### 1.5 Clasificación de variables: teoría vs. dtypes de Pandas

Se clasifica cada variable según la teoría estadística y se contrasta con el tipo de dato (*dtype*) que Pandas asignó automáticamente al leer el CSV. La columna `coincide` indica si el dtype es coherente con el tipo teórico (numérico para las cuantitativas, texto o categoría para las cualitativas).

In [ ]:
# Clasificación teórica de cada variable
tipo_teorico = {
    "customerID":       "Identificador",
    "gender":           "Cualitativa nominal",
    "SeniorCitizen":    "Cualitativa nominal",
    "Partner":          "Cualitativa nominal",
    "Dependents":       "Cualitativa nominal",
    "tenure":           "Cuantitativa discreta",
    "PhoneService":     "Cualitativa nominal",
    "MultipleLines":    "Cualitativa nominal",
    "InternetService":  "Cualitativa nominal",
    "OnlineSecurity":   "Cualitativa nominal",
    "OnlineBackup":     "Cualitativa nominal",
    "DeviceProtection": "Cualitativa nominal",
    "TechSupport":      "Cualitativa nominal",
    "StreamingTV":      "Cualitativa nominal",
    "StreamingMovies":  "Cualitativa nominal",
    "Contract":         "Cualitativa ordinal",
    "PaperlessBilling": "Cualitativa nominal",
    "PaymentMethod":    "Cualitativa nominal",
    "MonthlyCharges":   "Cuantitativa continua",
    "TotalCharges":     "Cuantitativa continua",
    "Churn":            "Cualitativa nominal (objetivo)",
}

clasificacion = pd.DataFrame({
    "variable": df.columns,
    "tipo_teorico": [tipo_teorico[c] for c in df.columns],
    "dtype_pandas": df.dtypes.astype(str).values,
    "valores_unicos": df.nunique().values,
})

# ¿El dtype es coherente con el tipo teórico?
def es_coherente(fila):
    es_numerico = pd.api.types.is_numeric_dtype(df[fila["variable"]])
    if fila["tipo_teorico"].startswith("Cuantitativa"):
        return es_numerico
    return not es_numerico   # cualitativas e identificador deberían ser texto/categoría

clasificacion["coincide"] = clasificacion.apply(es_coherente, axis=1)
clasificacion

In [ ]:
# Variables donde la teoría y el dtype de Pandas NO coinciden
clasificacion[~clasificacion["coincide"]]

**Interpretación:**

La mayoría de las variables son **cualitativas nominales** (servicios contratados y datos demográficos con categorías como `Yes`/`No`), y Pandas las leyó como texto (`object`; en versiones recientes de Pandas aparece como `str`), lo cual es coherente.

- **`Contract`** se clasifica como **ordinal** porque sus categorías tienen un orden natural según la duración del compromiso: *Month-to-month* < *One year* < *Two year*.
- **`tenure`** (meses de antigüedad) es **cuantitativa discreta**: toma valores enteros contables. **`MonthlyCharges`** y **`TotalCharges`** son **cuantitativas continuas** (importes de dinero con decimales).
- **`customerID`** es un **identificador**: distingue a cada cliente pero no aporta información analítica, por lo que se excluirá del modelado.

Se detectan **dos discrepancias** entre la teoría y los dtypes:

1. **`SeniorCitizen`** es conceptualmente cualitativa (el cliente es o no es adulto mayor), pero Pandas la leyó como `int64` porque viene **codificada como 0/1**. Calcular su promedio o desvío no tiene sentido como medida de una cantidad, por lo que se convertirá a categoría.
2. **`TotalCharges`** es cuantitativa continua, pero Pandas la leyó como **texto**. Esto indica que la columna contiene algún valor no numérico que impidió la conversión automática. Se investiga y corrige en la sección 3.

---
# 2. A — Manipulación de datos
**Responsable:** _a definir_

### 2.1 Selección y filtrado con múltiples condiciones
_Usar operadores `&` y `|`. Cada filtro debe responder una pregunta de negocio concreta._

In [ ]:
# Ejemplo: clientes con contrato mes a mes Y antigüedad menor a 12 meses
# filtro = df[(df["Contract"] == "Month-to-month") & (df["tenure"] < 12)]

# TODO: agregar filtros propios con & y |

### 2.2 Agregación con `groupby`
_Segmentar el negocio combinando varias agregaciones (promedio, conteo, suma)._

In [ ]:
# Ejemplo de estructura:
# df.groupby("Contract").agg(
#     clientes=("customerID", "count"),
#     cargo_mensual_prom=("MonthlyCharges", "mean"),
#     ...
# )

# TODO

### 2.3 Transformación básica
_Renombrar columnas, eliminar atributos irrelevantes (justificar) y crear columnas calculadas con operaciones aritméticas entre variables._

In [ ]:
# TODO: renombrado de columnas (si el equipo decide traducirlas o normalizarlas)

# TODO: eliminación de atributos irrelevantes (ej.: identificadores), con justificación en markdown

# TODO: columnas calculadas (ej.: relación entre TotalCharges, MonthlyCharges y tenure)

---
# 3. A — Limpieza y calidad de datos
**Responsable:** _a definir_

### 3.1 Valores faltantes y cadenas anómalas
_Detectar nulos explícitos y cadenas vacías o con espacios (caso conocido: `TotalCharges`). Decidir y justificar: imputar o eliminar._

In [ ]:
# Nulos explícitos por columna
df.isna().sum()

In [ ]:
# TODO: detectar cadenas vacías o solo espacios en columnas de texto
# TODO: analizar QUIÉNES son esos registros antes de decidir qué hacer con ellos

### 3.2 Adecuación de tipos de datos
_Convertir formalmente los dtypes que no corresponden y mostrar el antes y el después._

In [ ]:
# TODO: conversión de tipos (ej.: pd.to_numeric, astype("category"), etc.)

### 3.3 Valores atípicos (outliers)
_Identificar outliers con boxplots y con el método IQR en `tenure`, `MonthlyCharges` y `TotalCharges`. Justificar si se tratan o se conservan._

In [ ]:
def outliers_iqr(serie: pd.Series) -> pd.Series:
    """Devuelve una máscara booleana con los valores fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]."""
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)

# TODO: aplicar a las variables numéricas (una vez corregidos los dtypes) y graficar boxplots

---
# 4. B — Análisis exploratorio y estadística aplicada
**Responsable:** _a definir_

### 4.1 Análisis cuantitativo
_Medidas de tendencia central y dispersión de `tenure`, `MonthlyCharges` y `TotalCharges`, con interpretación._

In [ ]:
# TODO: describe() + mediana, moda, rango, desvío, coeficiente de variación

### 4.2 Correlación lineal (Pearson / Spearman)
_Calcular ambas, comparar y explicar cuándo conviene cada una._

In [ ]:
# TODO: df[[...]].corr(method="pearson") y method="spearman"

### 4.3 Análisis categórico
_Frecuencias absolutas y relativas, y tablas de contingencia de variables sociodemográficas y de contratación contra `Churn`._

In [ ]:
# TODO: value_counts() y value_counts(normalize=True)
# TODO: pd.crosstab(..., df["Churn"], normalize="index")

### 4.4 Contraste de hipótesis planteadas
_Volver a las hipótesis H1–H4 de la sección 1.3 e indicar si los datos las respaldan._

---
# 5. B — Visualización de datos
**Responsable:** _a definir_

> Cada gráfico va seguido de una celda con su interpretación orientada al negocio.

### 5.1 Distribución de `tenure` y `MonthlyCharges` (histograma + KDE)

In [ ]:
# TODO

_Interpretación:_

### 5.2 Matriz de correlación (heatmap)

In [ ]:
# TODO

_Interpretación:_

### 5.3 Tasa de Churn por tipo de contrato

In [ ]:
# TODO

_Interpretación:_

### 5.4 Tasa de Churn por método de pago

In [ ]:
# TODO

_Interpretación:_

---
# 6. C — Preprocesamiento para modelado
**Responsable:** _a definir_

> **Regla anti data leakage:** primero se hace la partición train/test; los encoders y scalers se ajustan (`fit`) **solo con train** y después se aplican (`transform`) a test. Un `Pipeline` con `ColumnTransformer` garantiza esto automáticamente.

### 6.1 Definición de X e y

In [ ]:
# TODO: definir features (X) y objetivo (y), excluyendo identificadores

### 6.2 Partición estratificada 80/20

In [ ]:
# TODO: train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

### 6.3 Codificación y escalado
_Justificar qué variables van con One-Hot y cuáles con Ordinal, y por qué StandardScaler o MinMaxScaler._

In [ ]:
# TODO: ColumnTransformer con OneHotEncoder / OrdinalEncoder y StandardScaler / MinMaxScaler

---
# 7. C — Predicción numérica: regresión lineal simple
**Responsable:** _a definir_

_Estimar una variable continua (ej.: `TotalCharges`) a partir de **una** variable predictora. Justificar la elección del predictor._

In [ ]:
# TODO: entrenar LinearRegression con una sola feature
# TODO: evaluar con r2_score y mean_absolute_error sobre test

_Interpretación de coeficientes y métricas:_

---
# 8. C — Predicción categórica: clasificación de Churn
**Responsable:** _a definir_

_Entrenar **un** modelo elemental (Regresión Logística o Árbol de Decisión) y justificar la elección._

In [ ]:
# TODO: Pipeline(preprocesador + clasificador), fit sobre train
# TODO: predecir sobre test

In [ ]:
# TODO: matriz de confusión (ConfusionMatrixDisplay) y accuracy_score

_Interpretación: qué significan los falsos negativos y falsos positivos para el negocio, y por qué la accuracy por sí sola puede ser engañosa con clases desbalanceadas._

---
# 9. Conclusiones
**Responsable:** _todo el equipo_

### 9.1 Hallazgos principales
_..._

### 9.2 Recomendaciones de negocio
_..._

### 9.3 Limitaciones y próximos pasos
_..._